In [1]:
import pandas as pd
from dotenv import load_dotenv
import os
from typing import List, Tuple

from tqdm import tqdm

from concurrent.futures import ThreadPoolExecutor, as_completed

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage

In [2]:
df_final = pd.read_csv('sampled_questions - agg.csv')
df_final_labels = df_final.loc[:, ['question', 'final']]
df_final_labels.head()

,question,final
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",0
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1


In [3]:
load_dotenv('/Users/egorsmirnov/Desktop/.env')
parser = StrOutputParser()

models = ['gpt-oss-20b', 'gpt-oss-120b', 'llama-4-scout', 'llama-4-maverick', 'qwen3-14b', 'qwen3-32b', 'qwen3-235b-a22b']

API_KEY = os.getenv("BOTHUB_API_KEY")
url = 'https://bothub.chat/api/v2/openai/v1'

llm = ChatOpenAI(api_key=API_KEY, base_url=url, model='qwen3-14b')

SYSTEM_PROMPT = """
Ты — классификатор вопросов. Твоя задача: определить, является ли вопрос ИНФОРМАТИВНЫМ или НЕИНФОРМАТИВНЫМ.

Верни строго одно число:
1 — если вопрос информативный
0 — если вопрос неинформативный

Определения:ИНФОРМАТИВНЫЙ ВОПРОС — это вопрос, который имеет явную цель получить новую, недостающую информацию. Человек хочет узнать факт, способ действия, местоположение, процедуру, контакт, цену, наличие, правила, инструкции, рекомендации или другую конкретную информацию.  
Признаки:
- вопрос направлен на восполнение знаний
- содержит запрос на факты, инструкции, опыт, советы
- человек действительно хочет что‑то узнать

НЕИНФОРМАТИВНЫЙ ВОПРОС — это вопрос, который НЕ направлен на получение новой информации.  
Признаки:
- риторический вопрос
- продолжение диалога без запроса знаний
- выражение эмоций, иронии, шутки
- общеизвестный факт, не требующий ответа
- вопрос, не предполагающий полезного ответа

Примеры ИНФОРМАТИВНЫХ вопросов:
- «Где купить свежие морепродукты рядом с Убудом?»
- «Как записаться на маникюр за 5300 ₽?»
- «Кто может помыть окна в ЖК? Поделитесь контактами.»
- «Где посмотреть кроватку ребёнку после колыбельки?»
- «Есть тут фотограф для семейной фотосессии?»

Примеры НЕИНФОРМАТИВНЫХ вопросов:
- «Я могу написать тебе в лс?»
- «А зачем это строить»
- «Бюрки где?» (если контекст отсутствует)
- «А под что маскировать?»
- «Рей или Каору?»
- «Вам таблетки или травы?»

Формат ответа:
Верни только 0 или 1. Никаких пояснений, текста или комментариев.
"""


messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=str(df_final_labels.iloc[0, 0]))]

llm_response = llm.invoke(messages)
result = parser.invoke(llm_response)


print(result)

1


In [4]:
class LLM_classifier:
    
    def __init__(self, model, API_KEY, SYSTEM_PROMPT, url):
        self.parser = StrOutputParser()
        self.model = model
        self.API_KEY = API_KEY
        self.SYSTEM_PROMPT = SYSTEM_PROMPT
        self.url = url
        
    def classify_question(self, llm, question) -> str:
        message = [SystemMessage(content=self.SYSTEM_PROMPT), HumanMessage(content=str(question))
                  ]
        llm_response = llm.invoke(message)
        return self.parser.invoke(llm_response)
        

    def classify_dataframe(self, df_final_labels, model_name) -> pd.DataFrame:
        result = pd.DataFrame()
        result['question'] = df_final_labels['question']

        llm = ChatOpenAI(api_key=self.API_KEY, base_url=self.url, model=model_name)
        labels = []
        for question in tqdm(df_final_labels['question'], desc=f'{model_name}'):
            label = self.classify_question(llm, question)
            labels.append(label)
            
        result[model_name] = labels
        return result


In [5]:
SYSTEM_PROMPT = """
Ты — классификатор вопросов. Твоя задача: определить, является ли вопрос ИНФОРМАТИВНЫМ или НЕИНФОРМАТИВНЫМ.

Верни строго одно число:
1 — если вопрос информативный
0 — если вопрос неинформативный

Определения:ИНФОРМАТИВНЫЙ ВОПРОС — это вопрос, который имеет явную цель получить новую, недостающую информацию. Человек хочет узнать факт, способ действия, местоположение, процедуру, контакт, цену, наличие, правила, инструкции, рекомендации или другую конкретную информацию.  
Признаки:
- вопрос направлен на восполнение знаний
- содержит запрос на факты, инструкции, опыт, советы
- человек действительно хочет что‑то узнать

НЕИНФОРМАТИВНЫЙ ВОПРОС — это вопрос, который НЕ направлен на получение новой информации.  
Признаки:
- риторический вопрос
- продолжение диалога без запроса знаний
- выражение эмоций, иронии, шутки
- общеизвестный факт, не требующий ответа
- вопрос, не предполагающий полезного ответа

Примеры ИНФОРМАТИВНЫХ вопросов:
- «Где купить свежие морепродукты рядом с Убудом?»
- «Как записаться на маникюр за 5300 ₽?»
- «Кто может помыть окна в ЖК? Поделитесь контактами.»
- «Где посмотреть кроватку ребёнку после колыбельки?»
- «Есть тут фотограф для семейной фотосессии?»

Примеры НЕИНФОРМАТИВНЫХ вопросов:
- «Я могу написать тебе в лс?»
- «А зачем это строить»
- «Бюрки где?» (если контекст отсутствует)
- «А под что маскировать?»
- «Рей или Каору?»
- «Вам таблетки или травы?»

Формат ответа:
Верни только 0 или 1. Никаких пояснений, текста или комментариев.
"""

In [6]:
models = ['gpt-oss-20b', 'gpt-oss-120b', 'llama-4-scout', 'llama-4-maverick', 'qwen3-14b', 'qwen3-32b', 'qwen3-235b-a22b']
API_KEY = os.getenv("BOTHUB_API_KEY")
url = 'https://bothub.chat/api/v2/openai/v1'

#classifier = LLM_classifier(models, API_KEY, SYSTEM_PROMPT, url)
#results_df = classifier.classify_dataframe(df_final_labels.loc[:11, ['question']])

In [7]:
classifier = LLM_classifier(models, API_KEY, SYSTEM_PROMPT, url)
results_df_llama_4_scout = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'llama-4-scout')

llama-4-scout:  76%|███████████████████▋      | 249/329 [10:53<03:29,  2.62s/it]


PermissionDeniedError: Error code: 403 - {'status': 'error', 'error': {'message': 'Balance is below zero', 'code': 'NOT_ENOUGH_TOKENS'}}

In [11]:
results_df_llama_4_scout

NameError: name 'results_df_llama_4_scout' is not defined

In [ ]:
results_df.to_csv('llm_classifications.csv')